<a href="https://colab.research.google.com/github/sabharwal-monish/LLM/blob/main/LLM_Training_in_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch

In [ ]:
!pip install "deeplake<4"

# Load Dataset from Deep Lake

In [ ]:
import deeplake

ds = deeplake.load('hub://activeloop/openwebtext-train', read_only=True)
ds_val = deeplake.load('hub://activeloop/openwebtext-val', read_only=True)

print("Connection Successful!")
print(ds)
print(ds[0].text.text())

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

  def __init__(self, ds):
    self.ds = ds

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, idx):
    tokenized_text = tokenizer(
        self.ds.text[idx].text(),
        truncation = True,
        max_length = 512,
        padding = 'max_length',
        return_tensors = 'pt'
    )

    tokenized_text = tokenized_text['input_ids'][0]

    sample = {'input_ids': tokenized_text, 'labels': tokenized_text }
    return sample



In [ ]:
myTrainingLoader = MyDataset(ds)
myValidationLoader = MyDataset(ds_val)

# Load the Model

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained('gpt2')
print(config)

In [ ]:
from transformers import GPT2LMHeadModel
from accelerate import Accelerator

model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f'GPT2 Size:{model_size/1e6:.1f} M Parameters')


# Training

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="GPT2-scratch-openwebtext",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,


    max_steps=500,
    num_train_epochs=2,


    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,


    bf16=False,
    fp16=True,

    ddp_find_unused_parameters=False,
    run_name="GPT2-scratch-openwebtext",
    report_to="none"
)

print("Arguments loaded successfully!")

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=myTrainingLoader,
    eval_dataset=myValidationLoader,
)

In [ ]:
from transformers import Trainer
trainer.train()

# Inference

In [ ]:

save_path = "./my_final_gpt2"

print("Saving model to disk...")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Save complete!")




from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

print("Loading pipeline...")
pipe = pipeline("text-generation",
                model=save_path,
                tokenizer=tokenizer,
                device=device)


print("Generating text...")
txt = "The house prices dropped down"
completion = pipe(txt, num_return_sequences=1)
print(completion)